🧠 Objective

The goal of this notebook is to build a simple, rule-based product recommendation system using transaction data.

This analysis focuses on:

Identifying products frequently bought together

Creating basic cross-selling recommendations

Translating analytical results into business actions

📦 Dataset

Online Retail Dataset (UK-based e-commerce)

Transactions from 2010–2011

Cleaned dataset loaded at the start of this notebook

In [ ]:
# =============================================================================
# Notebook 04 – Market Basket Analysis, Recommendations & Final Report
# =============================================================================

# ── 1. Imports & Setup ───────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
import os

# Set seaborn style for nicer plots
sns.set_style("whitegrid")

# ── 2. Load Cleaned Data ─────────────────────────────────────────────────────
print("Loading data...")
df_clean = pd.read_csv("../data/online-retail-cleaned.csv", encoding="ISO-8859-1")
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"Data loaded: {df_clean.shape[0]:,} rows")
print(f"Unique customers: {df_clean['CustomerID'].nunique():,}")
print(f"Unique products (StockCode): {df_clean['StockCode'].nunique():,}")
print(f"Unique descriptions: {df_clean['Description'].nunique():,}")
print(f"Date range: {df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}")
print(f"Total revenue: £{df_clean['Revenue'].sum():,.2f}\n")

# ── 3. Prepare Transaction Basket ────────────────────────────────────────────
print("Creating basket matrix (this may take a moment)...")
basket = (df_clean.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().fillna(0)
          .applymap(lambda x: 1 if x > 0 else 0))

print(f"Basket shape: {basket.shape}\n")

# ── 4. Generate Frequent Itemsets & Association Rules ─────────────────────────
print("Running Apriori (min_support=0.01)...")
frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)

print("Generating rules (min_confidence=0.3)...")
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)

# Filter strong rules
rules = rules[rules['lift'] > 2].sort_values('lift', ascending=False)

print(f"Strong rules found (lift > 2): {len(rules):,}\n")
print("Top 8 rules by lift:")
display(rules.head(8))

# ── 5. Recommendation Function ───────────────────────────────────────────────
def recommend_products(product_name=None, customer_id=None, top_n=5):
    recommendations = []
    
    if product_name:
        product_name = str(product_name).strip().upper()
        filtered = rules[rules['antecedents'].apply(lambda x: product_name in x)]
        for _, row in filtered.head(top_n).iterrows():
            recommendations.append({
                'Type': 'Product-based',
                'Antecedent': list(row['antecedents']),
                'Recommended': list(row['consequents']),
                'Confidence': round(row['confidence'], 3),
                'Lift': round(row['lift'], 2)
            })
    
    elif customer_id:
        purchases = df_clean[df_clean['CustomerID'] == customer_id]['Description'].unique()
        if len(purchases) == 0:
            return [{"Type": "Note", "Message": f"No purchases found for CustomerID {customer_id}"}]
        for p in purchases[:3]:  # limit to last 3 items to avoid explosion
            filtered = rules[rules['antecedents'].apply(lambda x: p in x)]
            for _, row in filtered.head(top_n).iterrows():
                recommendations.append({
                    'Type': 'Customer-based',
                    'Based on': p,
                    'Recommended': list(row['consequents']),
                    'Confidence': round(row['confidence'], 3),
                    'Lift': round(row['lift'], 2)
                })
    
    else:
        # Popular fallback
        top_products = df_clean['Description'].value_counts().head(top_n)
        for prod, count in top_products.items():
            recommendations.append({
                'Type': 'Popular Product',
                'Recommended': prod,
                'Purchase Count': count
            })
    
    return pd.DataFrame(recommendations) if recommendations else "No recommendations found"

# ── 6. Quick Tests ────────────────────────────────────────────────────────────
print("\nExample 1: Product-based recommendation")
popular_item = "WHITE HANGING HEART T-LIGHT HOLDER"
recs_product = recommend_products(product_name=popular_item)
print(f"Recommendations for '{popular_item}':")
display(recs_product)

print("\nExample 2: Popular products fallback")
recs_popular = recommend_products(top_n=8)
display(recs_popular)

# ── 7. Co-Purchase Heatmap (Top 20 products) ─────────────────────────────────
if 'basket' in globals():
    print("\nGenerating co-purchase heatmap for top 20 products...")
    top_products = basket.sum().sort_values(ascending=False).head(20).index
    product_pairs = basket.T.dot(basket)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(product_pairs.loc[top_products, top_products], 
                cmap="YlOrRd", annot=False, fmt=".0f", 
                cbar_kws={'label': 'Co-purchase count'})
    plt.title("Co-Purchase Heatmap – Top 20 Products")
    plt.tight_layout()
    plt.show()
else:
    print("Basket matrix not found – skipping heatmap.")

# ── 8. Customer Summary ──────────────────────────────────────────────────────
print("\nCreating customer summary...")
customer_summary = df_clean.groupby('CustomerID').agg({
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'Revenue': 'sum',
    'Description': lambda x: len(set(x))
}).rename(columns={
    'InvoiceNo': 'Total_Orders',
    'Quantity': 'Total_Items',
    'Revenue': 'Total_Revenue',
    'Description': 'Unique_Products'
})

customer_summary['Avg_Order_Value'] = (
    customer_summary['Total_Revenue'] / customer_summary['Total_Orders']
).round(2)

print("Customer summary stats:")
display(customer_summary.describe().round(1))

# ── 9. Save Results ──────────────────────────────────────────────────────────
os.makedirs('../results', exist_ok=True)

# Popular products
popular_products = df_clean['Description'].value_counts().head(20)
popular_products.to_csv('../results/popular_products.csv', header=['Purchase_Count'])

# Customer summary
customer_summary.to_csv('../results/customer_summary.csv')

# Top rules
rules.head(50).to_csv('../results/top_association_rules.csv', index=False)

print("\nFiles saved:")
print("  • ../results/popular_products.csv")
print("  • ../results/customer_summary.csv")
print("  • ../results/top_association_rules.csv")

# ── 10. Final Summary Report (text file) ─────────────────────────────────────
summary_text = f"""ONLINE RETAIL ANALYTICS & RECOMMENDATION SYSTEM
==============================================
Date: January 2026

DATA OVERVIEW
------------
Total transactions: {df_clean.shape[0]:,}
Unique customers:   {df_clean['CustomerID'].nunique():,}
Unique products:    {df_clean['StockCode'].nunique():,}
Date range:         {df_clean['InvoiceDate'].min().date()} – {df_clean['InvoiceDate'].max().date()}
Total revenue:      £{df_clean['Revenue'].sum():,.2f}

KEY METRICS
-----------
Avg orders / customer:    {customer_summary['Total_Orders'].mean():.1f}
Avg revenue / customer:   £{customer_summary['Total_Revenue'].mean():,.2f}
Avg order value:          £{customer_summary['Avg_Order_Value'].mean():,.2f}
Strong rules (lift > 2):  {len(rules):,}

TOP 5 PRODUCTS
--------------
{popular_products.head(5).to_string()}

BUSINESS RECOMMENDATIONS
-----------------------
• Create bundles from high-lift rules (e.g. "Frequently bought together")
• Show real-time recommendations in cart / product pages
• Target high-value customers with personalized emails
• Promote top sellers + seasonal clusters
• Monitor stock of frequently co-purchased items

Generated with Apriori (supp=0.01, conf≥0.3, lift>2)
"""

with open('../results/final_project_report.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print("\n✅ Final report created: ../results/final_project_report.txt")
print("You can now open it or include it in your presentation.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

...

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Loading data...
Data loaded: 397,884 rows
Unique customers: 4,338
Unique products (StockCode): 3,665
Unique descriptions: 3,877
Date range: 2010-12-01 → 2011-12-09
Total revenue: £8,911,407.90

Creating basket matrix (this may take a moment)...


/var/folders/6l/gvvhxtl12fs_5yk6w39y4yh40000gq/T/ipykernel_30022/2679931037.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: 1 if x > 0 else 0))


Basket shape: (18532, 3877)

Running Apriori (min_support=0.01)...


/opt/anaconda3/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
